# CPU tokenizer-throughput benchmark

**Purpose:** Present committed CPU tokenizer-throughput observations; this is tokenization throughput only, not model quality, GPU latency, or cost.

**Inputs (committed result/provenance files read):**
- [`results/tables/runtime_throughput.csv`](../results/tables/runtime_throughput.csv)
- [`results/tables/runtime_summary.csv`](../results/tables/runtime_summary.csv)
- [`results/tables/fig06_runtime.csv`](../results/tables/fig06_runtime.csv)

**Outputs:** Rendered CPU tokenizer-throughput summaries in this notebook only; no files are written.

**Expected runtime:** ~3 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## CPU tokenizer-throughput benchmark

Authoritative files: [`runtime_throughput.csv`](../results/tables/runtime_throughput.csv), [`runtime_summary.csv`](../results/tables/runtime_summary.csv).

In [2]:
throughput = load_csv('results/tables/runtime_throughput.csv')
runtime_summary = load_csv('results/tables/runtime_summary.csv')
fig_runtime = load_csv('results/tables/fig06_runtime.csv')

In [3]:
if not throughput.empty:
    display(throughput.sort_values('tokenizer_id'))
else:
    note_missing('results/tables/runtime_throughput.csv')

,tokenizer_id,mb_per_second_median,mb_per_second_p95,load_seconds
4,azure-bytebpe-16384,5.466461,5.769760,0.0
5,azure-bytebpe-32768,5.469546,5.762162,0.0
3,azure-bytebpe-8192,5.438783,5.714801,0.0
1,general-bytebpe-16384,5.162949,5.440937,0.0
2,general-bytebpe-32768,5.186244,5.483565,0.0
0,general-bytebpe-8192,5.129768,5.403087,0.0
7,mixed-bytebpe-16384,5.449517,5.738162,0.0
8,mixed-bytebpe-32768,5.428504,5.744566,0.0
6,mixed-bytebpe-8192,5.412361,5.676226,0.0
10,tiktoken-cl100k_base,19.834159,21.309762,0.0


In [4]:
if not runtime_summary.empty:
    display(runtime_summary)
else:
    note_missing('results/tables/runtime_summary.csv')

,track,min,max
0,A,5.129768,5.469546
1,B,11.607775,19.834159


In [5]:
if not fig_runtime.empty:
    display(fig_runtime.sort_values(['track', 'tokenizer_id']))
else:
    note_missing('results/tables/fig06_runtime.csv')

,tokenizer_id,tokens_per_1000_bytes,mb_per_second_median,track
4,azure-bytebpe-16384,240.180213,5.466461,A (controlled)
5,azure-bytebpe-32768,232.776060,5.469546,A (controlled)
3,azure-bytebpe-8192,256.210308,5.438783,A (controlled)
1,general-bytebpe-16384,379.295492,5.162949,A (controlled)
2,general-bytebpe-32768,360.377429,5.186244,A (controlled)
0,general-bytebpe-8192,403.225451,5.129768,A (controlled)
7,mixed-bytebpe-16384,249.523101,5.449517,A (controlled)
8,mixed-bytebpe-32768,238.815804,5.428504,A (controlled)
6,mixed-bytebpe-8192,270.921452,5.412361,A (controlled)
10,tiktoken-cl100k_base,217.391601,19.834159,"B (tiktoken, observational)"
